# Engagement-filtering audit — Figure 2 / Supp Fig 2 behavior metrics

**Purpose**
1. Report which engagement-related columns actually exist in each saved table (resolves the
   `engaged` vs `engagement_state` question and tells you which code path produced the tables).
2. Quantify how much limiting to *engaged* trials changes the reported metrics, **per experience level**,
   to reveal whether engaged-only filtering introduces a condition-dependent bias.

Run in the `visual_behavior_sdk` env. Reads only the already-saved tables — regenerates nothing.

**The bias being probed:** engagement is defined as rolling reward_rate > 2 rewards/min, and reward_rate
is essentially (recent hits)/time. So "engaged" epochs are by construction the high-hit-rate epochs.
The columns below let you measure how large that inflation is and whether it differs across
Familiar / Novel / Novel+ — which is what would distort a cross-experience comparison.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from visual_behavior.data_access import loading
import visual_behavior.visualization.utils as utils

%matplotlib inline
sns.set_context('notebook')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

colors = utils.get_experience_level_colors()
exp_order = ['Familiar', 'Novel', 'Novel +']

behavior_metrics_dir = loading.get_performance_metrics_dir()
cache_dir = loading.get_platform_analysis_cache_dir()
metadata_tables_dir = loading.get_metadata_tables_dir()

platform_experiments = pd.read_csv(
    os.path.join(metadata_tables_dir, 'platform_paper_ophys_experiments_table.csv'), index_col=0)
exp_lookup = (platform_experiments[['behavior_session_id', 'experience_level', 'cell_type']]
              .drop_duplicates('behavior_session_id').set_index('behavior_session_id'))

def _try_load_hdf(path):
    return pd.read_hdf(path, key='data') if os.path.exists(path) else None

tables = {}
tables['sdk'] = _try_load_hdf(os.path.join(behavior_metrics_dir, 'sdk_all_sessions.h5'))
tables['stimulus_based_all'] = _try_load_hdf(os.path.join(behavior_metrics_dir, 'stimulus_based_all_sessions.h5'))
tables['stimulus_based_engaged'] = _try_load_hdf(os.path.join(cache_dir, 'stimulus_based_engaged_only_all_sessions.h5'))
p = os.path.join(behavior_metrics_dir, 'platform_behavior_stats_engaged.csv')
tables['platform_behavior_stats_engaged'] = pd.read_csv(p, index_col=0) if os.path.exists(p) else None

def _ensure_bsid(df):
    if df is None:
        return None
    if 'behavior_session_id' not in df.columns:
        df = df.reset_index()
        if 'behavior_session_id' not in df.columns and 'index' in df.columns:
            df = df.rename(columns={'index': 'behavior_session_id'})
    return df

def _add_experience(df):
    df = _ensure_bsid(df)
    if df is None or 'behavior_session_id' not in df.columns:
        return df
    return df.merge(exp_lookup, on='behavior_session_id', how='left')

def _order(idx):
    o = [e for e in exp_order if e in list(idx)]
    return o if o else list(idx)

def _short(labels):
    return [{'Familiar': 'F', 'Novel': 'N', 'Novel +': 'N+'}.get(l, l) for l in labels]

## 1. Column inventory — which engagement column actually got written?

In [ ]:
engage_tokens = ['engage', 'engaged', 'reward_rate', 'fraction']
for name, df in tables.items():
    if df is None:
        print(f'\n[{name}]  -- FILE NOT FOUND --'); continue
    cols = list(df.columns)
    engage_cols = [c for c in cols if any(tok in c.lower() for tok in engage_tokens)]
    print(f'\n[{name}]  shape={df.shape}')
    print(f'   engagement-related columns: {engage_cols if engage_cols else "NONE"}')
    print(f'   all columns: {cols}')

## 2. SDK table: engaged vs whole-session, by experience level

These come from one table (the SDK returns both versions of each metric), so it's a clean
within-session comparison.

In [ ]:
sdk = _add_experience(tables['sdk'])
if {'engaged_trial_count', 'go_trial_count'}.issubset(sdk.columns):
    sdk['frac_engaged_go'] = sdk['engaged_trial_count'] / sdk['go_trial_count']

pairs = [('mean_hit_rate', 'mean_hit_rate_engaged'),
         ('mean_false_alarm_rate', 'mean_false_alarm_rate_engaged'),
         ('mean_dprime', 'mean_dprime_engaged'),
         ('max_dprime', 'max_dprime_engaged')]
cols_present = [c for pr in pairs for c in pr if c in sdk.columns]
extra = [c for c in ['frac_engaged_go'] if c in sdk.columns]
order = _order(sdk['experience_level'].unique())
sdk_summary = sdk.groupby('experience_level')[cols_present + extra].mean(numeric_only=True).reindex(order)
print('Group means:')
display(sdk_summary.round(3).T)

### Diagnostic plots — SDK

In [ ]:
plot_pairs = [(w, e) for (w, e) in pairs if w in sdk.columns and e in sdk.columns]
order = _order(sdk['experience_level'].unique())
x = np.arange(len(order)); bw = 0.38

fig, axes = plt.subplots(1, len(plot_pairs), figsize=(3.6 * len(plot_pairs), 3.8))
if len(plot_pairs) == 1: axes = [axes]
for ax, (whole, eng) in zip(axes, plot_pairs):
    gm_w = sdk.groupby('experience_level')[whole].mean().reindex(order)
    gm_e = sdk.groupby('experience_level')[eng].mean().reindex(order)
    se_w = sdk.groupby('experience_level')[whole].sem().reindex(order)
    se_e = sdk.groupby('experience_level')[eng].sem().reindex(order)
    ax.bar(x - bw/2, gm_w.values, bw, yerr=se_w.values, color='lightgray', edgecolor='k', label='whole session')
    ax.bar(x + bw/2, gm_e.values, bw, yerr=se_e.values, color=colors[:len(order)], edgecolor='k', label='engaged only')
    ax.set_xticks(x); ax.set_xticklabels(_short(order))
    ax.set_title(whole.replace('mean_', '').replace('_', ' '))
axes[0].legend(fontsize=9, frameon=False)
plt.suptitle('SDK metrics: whole-session (gray) vs engaged-only (color)', y=1.04)
plt.tight_layout()
plt.show()

In [ ]:
# Condition-dependent inflation: (engaged - whole) by experience. Diverging lines = biased F/N/N+ comparison.
fig, ax = plt.subplots(figsize=(6, 4))
for whole, eng in plot_pairs:
    d = (sdk[eng] - sdk[whole]).groupby(sdk['experience_level']).mean().reindex(order)
    ax.plot(range(len(order)), d.values, '-o', label=eng.replace('_engaged', '').replace('mean_', ''))
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(range(len(order))); ax.set_xticklabels(_short(order))
ax.set_ylabel('engaged  -  whole session')
ax.set_title('Engagement-filter inflation by experience\n(non-flat / diverging lines = condition-dependent bias)')
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 3. stimulus_based: engaged-only table vs all-trials table, matched by behavior_session_id

In [ ]:
all_df = _ensure_bsid(tables['stimulus_based_all'])
eng_df = _ensure_bsid(tables['stimulus_based_engaged'])
metric_cols = []
merged = None
if all_df is not None and eng_df is not None:
    metric_cols = [c for c in ['hit_rate', 'fa_rate', 'dprime_trial_corrected',
                               'dprime_non_trial_corrected', 'response_latency_mean']
                   if c in all_df.columns and c in eng_df.columns]
    merged = all_df[['behavior_session_id'] + metric_cols].merge(
        eng_df[['behavior_session_id'] + metric_cols], on='behavior_session_id', suffixes=('_all', '_engaged'))
    merged = merged.merge(exp_lookup, on='behavior_session_id', how='left')
    order2 = _order(merged['experience_level'].unique())
    delta = pd.DataFrame({c: (merged[f'{c}_engaged'] - merged[f'{c}_all']) for c in metric_cols})
    delta['experience_level'] = merged['experience_level'].values
    print('Per-session (engaged - all) delta, mean by experience level:')
    display(delta.groupby('experience_level').mean().reindex(order2).round(3))
else:
    print('stimulus_based tables not both available; skipping.')

### Diagnostic plots — stimulus_based & engagement fraction

In [ ]:
# Per-session engaged-vs-all delta for each metric (boxplots; >0 means engaged filter raises the metric)
if merged is not None and metric_cols:
    order2 = _order(merged['experience_level'].unique())
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(3.0 * len(metric_cols), 3.6))
    if len(metric_cols) == 1: axes = [axes]
    for ax, c in zip(axes, metric_cols):
        d = pd.DataFrame({'delta': merged[f'{c}_engaged'] - merged[f'{c}_all'],
                          'experience_level': merged['experience_level'].values})
        sns.boxplot(data=d, x='experience_level', y='delta', order=order2,
                    palette=colors[:len(order2)], fliersize=0, ax=ax)
        ax.axhline(0, color='k', lw=0.8)
        ax.set_xticklabels(_short(order2)); ax.set_xlabel('')
        ax.set_title(c, fontsize=10); ax.set_ylabel('engaged - all')
    plt.suptitle('stimulus_based: per-session engaged-minus-all delta', y=1.04)
    plt.tight_layout(); plt.show()

In [ ]:
# fraction_engaged by experience: if this differs across conditions, the engaged filter removes a
# different fraction of trials per condition -> the mechanism for condition-dependent bias.
fe_src = None
for src in [all_df, eng_df, _ensure_bsid(tables['platform_behavior_stats_engaged']), sdk]:
    if src is not None and 'fraction_engaged' in getattr(src, 'columns', []):
        fe_src = src; break
if fe_src is None and 'frac_engaged_go' in sdk.columns:
    fe_src = sdk.rename(columns={'frac_engaged_go': 'fraction_engaged'})

if fe_src is not None:
    fe = _add_experience(fe_src)
    order_fe = _order(fe['experience_level'].dropna().unique())
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.boxplot(data=fe, x='experience_level', y='fraction_engaged', order=order_fe,
                palette=colors[:len(order_fe)], fliersize=0, ax=ax)
    sns.stripplot(data=fe, x='experience_level', y='fraction_engaged', order=order_fe,
                  color='k', size=2, alpha=0.4, ax=ax)
    ax.set_xticklabels(_short(order_fe)); ax.set_xlabel('')
    ax.set_title('fraction_engaged by experience\n(differs => engaged filter removes different fraction per condition)')
    plt.tight_layout(); plt.show()
else:
    print('No fraction_engaged column found in any table.')

## How to read these

- **Bars (SDK):** colored (engaged-only) consistently taller than gray (whole-session) = the engaged
  filter inflates the metric. `max_dprime` should be the least affected (it's the whole-session peak).
- **Inflation lines:** if the (engaged - whole) line is *flat* across F/N/N+, the bias is a constant
  offset and the cross-experience comparison is preserved. If it **diverges**, engaged-only filtering
  distorts the F-vs-N comparison itself.
- **stimulus_based delta boxes:** same logic per session; watch for a larger positive shift in one
  experience level than another.
- **fraction_engaged:** if this differs across experience levels, that is the mechanism — engaged-only
  metrics confound "performance when engaged" with "propensity to stay engaged."

If the divergence is meaningful, prefer whole-session metrics (`max_dprime`, whole-session `mean_dprime`)
for the cross-experience comparison, and report `fraction_engaged` separately.